## Progress report #3 Visualisations

### Packages

In [1]:
import os
import geopandas as gpd
import pandas as pd
from shapely import wkt
from shapely.geometry import box
import matplotlib.pyplot as plt
import contextily as ctx
import numpy as np
import matplotlib.patches as mpatches
from tqdm import tqdm

C:\Users\white_rn\AppData\Local\Temp\ipykernel_14000\1553837205.py:2: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


### Settings

In [41]:
# Settings
dir_path_main= r'p:/11209821-cmems-global-sdb'
zoomed_list = [9, 10, 11]
sel_tile = 1
GEBCO_scale = 465 #m, actually 463 m but rounded up to the nearest 5 m 
crs = "EPSG:4326"

# File paths
file_path_tiles_csv = os.path.join(dir_path_main, '00_miscellaneous', 'AOI_polygons_world', 'df_boxes_world_dist_Z{}.csv'.format(zoomed_list[sel_tile]))
file_path_mask_parquet = os.path.join(dir_path_main, '00_miscellaneous', 'Feasibility_maps', '2024', 'gebco_2024_latminus2_merge_result.parquet')
file_path_stations_geojson = r'p:\11209821-cmems-global-sdb\01_intertidal\02_data\02_gtsm_files\gtsm_stations.geojson'
file_path_aois = [r'p:\11209821-cmems-global-sdb\00_miscellaneous\AOI_upscale\AOI_WestAustralia.geojson',
                  r'p:\11209821-cmems-global-sdb\00_miscellaneous\AOI_upscale\AOI_WestEurope.geojson']

# Read tiles
df_tiles = pd.read_csv(file_path_tiles_csv, index_col=0) # EPSG3857 standard
df_tiles['geometry'] = df_tiles['geometry'].apply(wkt.loads)
gdf_tiles = gpd.GeoDataFrame(df_tiles, geometry='geometry', crs="EPSG:3857")
gdf_tiles = gdf_tiles[gdf_tiles.area < 1e10] # Remove error tiles
gdf_tiles['distance_nearest_station'] = gdf_tiles['distance_nearest_station']/1000 # convert to km
gdf_tiles = gdf_tiles.to_crs(crs)

# Read mask
gdf_mask = gpd.read_parquet(file_path_mask_parquet)
gdf_mask = gdf_mask[gdf_mask["pixel_value"] == 3.0]
gdf_mask = gdf_mask.to_crs(crs)

# Read GTSM stations
gdf_stations = gpd.read_file(file_path_stations_geojson)
gdf_stations = gdf_stations[(gdf_stations.n_timesteps_2019 != 0) & (gdf_stations.n_timesteps_2020 != 0) & 
                            (gdf_stations.n_timesteps_2021 != 0) & (gdf_stations.n_timesteps_2022 != 0) & (gdf_stations.n_timesteps_2023 != 0)].reset_index(drop=True) # filter the empty stations
gdf_stations = gdf_stations.to_crs(crs)

# Read AOIs
gdf_aois = []
for file_path_aoi in file_path_aois:
    gdf_aoi = gpd.read_file(file_path_aoi)
    gdf_aois.append(gdf_aoi.to_crs(crs))
gdf_aois = gpd.GeoDataFrame(pd.concat(gdf_aois, ignore_index=True), crs=crs)

In [42]:
# Export tiles to parquet
gdf_tiles.to_parquet(os.path.join(dir_path_main, '00_miscellaneous', 'AOI_polygons_world', 'df_boxes_world_dist_Z{}.parquet'.format(zoomed_list[sel_tile])))

### Plot Data

In [ ]:
# Create the figure
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
for idx in range(len(gdf_aois)):
    # Get aoi
    gdf_aoi = gdf_aois.iloc[[idx]]
    
    # Get square box
    bounds = gdf_aoi.total_bounds
    size = max(bounds[2]-bounds[0], bounds[3]-bounds[1])
    center = [(bounds[0]+bounds[2])/2, (bounds[1]+bounds[3])/2]
    bounds = [center[0]-size/2, center[1]-size/2, center[0]+size/2, center[1]+size/2]
    gdf_box = gpd.GeoDataFrame({'geometry': [box(*bounds)]}, crs=crs)

    # Intersect with aoi
    gdf_tiles_aoi = gdf_tiles[gdf_tiles.intersects(gdf_box.unary_union)]
    gdf_mask_aoi = gdf_mask[gdf_mask.intersects(gdf_box.unary_union)]
    gdf_stations_aoi = gdf_stations[gdf_stations.intersects(gdf_box.unary_union)]
    
    # Plot aoi
    gdf_tiles_aoi.plot(ax=axs[idx], color='none', edgecolor='black', alpha=0.5)
    axs[idx].add_patch(mpatches.Rectangle((np.nan, np.nan), np.nan, np.nan, facecolor='none', edgecolor='black', alpha=0.5, label='Tiles'))
    gdf_mask_aoi.plot(ax=axs[idx], color='blue', edgecolor='none', alpha=0.5)
    axs[idx].add_patch(mpatches.Rectangle((np.nan, np.nan), np.nan, np.nan, facecolor='blue', edgecolor='none', alpha=0.5, label='Processing area'))
    gdf_stations_aoi.plot(ax=axs[idx], color='green', edgecolor='none', markersize=10, label='GTSM stations')
    axs[idx].legend()

    # Set limits
    axs[idx].set_xlim(bounds[0], bounds[2])
    axs[idx].set_ylim(bounds[1], bounds[3])
    ctx.add_basemap(axs[idx], source=ctx.providers.OpenStreetMap.Mapnik, crs=crs)

    # Set labels
    axs[idx].set_xlabel('Longitude [deg]')
    axs[idx].set_ylabel('Latitude [deg]')

In [ ]:
# Create the figure
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
for idx in range(len(gdf_aois)):
    # Get aoi
    gdf_aoi = gdf_aois.iloc[[idx]]
    
    # Get square box
    bounds = gdf_aoi.total_bounds
    size = max(bounds[2]-bounds[0], bounds[3]-bounds[1])
    center = [(bounds[0]+bounds[2])/2, (bounds[1]+bounds[3])/2]
    bounds = [center[0]-size/2, center[1]-size/2, center[0]+size/2, center[1]+size/2]
    gdf_box = gpd.GeoDataFrame({'geometry': [box(*bounds)]}, crs=crs)

    # Intersect with aoi
    gdf_tiles_aoi = gdf_tiles[gdf_tiles.intersects(gdf_box.unary_union)]
    gdf_mask_aoi = gdf_mask[gdf_mask.intersects(gdf_box.unary_union)]
    gdf_stations_aoi = gdf_stations[gdf_stations.intersects(gdf_box.unary_union)]
    
    # Plot aoi
    gdf_tiles_aoi.plot(ax=axs[idx], column='distance_nearest_station', legend=True, vmin=0, legend_kwds={'label': "Distance to nearest station [km]"}, cmap='viridis')
    
    # Set limits
    axs[idx].set_xlim(bounds[0], bounds[2])
    axs[idx].set_ylim(bounds[1], bounds[3])
    ctx.add_basemap(axs[idx], source=ctx.providers.OpenStreetMap.Mapnik, crs=crs)

    # Set labels
    axs[idx].set_xlabel('Longitude [deg]')
    axs[idx].set_ylabel('Latitude [deg]')

In [ ]:
# Create the figure
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
for idx in range(len(gdf_aois)):
    # Get aoi
    gdf_aoi = gdf_aois.iloc[[idx]]
    
    # Get square box
    bounds = gdf_aoi.total_bounds
    size = max(bounds[2]-bounds[0], bounds[3]-bounds[1])
    center = [(bounds[0]+bounds[2])/2, (bounds[1]+bounds[3])/2]
    bounds = [center[0]-size/2, center[1]-size/2, center[0]+size/2, center[1]+size/2]
    gdf_box = gpd.GeoDataFrame({'geometry': [box(*bounds)]}, crs="EPSG:3857")

    # Intersect with aoi
    gdf_tiles_aoi = gdf_tiles[gdf_tiles.intersects(gdf_box.unary_union)]
    gdf_mask_aoi = gdf_mask[gdf_mask.intersects(gdf_box.unary_union)]
    
    # Plot aoi
    gdf_tiles_aoi.plot(ax=axs[idx], column='area_perc_org', legend=True, vmin=0, legend_kwds={'label': "Intertidal coverage [%]"}, cmap='viridis')

    # Set limits
    axs[idx].set_xlim(bounds[0], bounds[2])
    axs[idx].set_ylim(bounds[1], bounds[3])
    ctx.add_basemap(axs[idx], source=ctx.providers.OpenStreetMap.Mapnik, crs=gdf_aoi.crs)

    # Set labels
    axs[idx].set_xlabel('Longitude [deg]')
    axs[idx].set_ylabel('Latitude [deg]')

In [ ]:
# Plot histogram  percentage
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

xlims = [0, 100]
gdf_tiles.hist(column='distance_nearest_station', ax=axs[0], bins=50, range=xlims, color='grey', edgecolor='black')
axs[0].set_xlim(xlims)
axs[0].set_xlabel('Distance to nearest station [km]')
axs[0].set_ylabel('Number of tiles [-]')
axs[0].set_title('')

xlims = [0, 50]
gdf_tiles.hist(column='area_perc_org', ax=axs[1], bins=50, range=xlims, color='grey', edgecolor='black')
axs[1].set_xlim(xlims)
axs[1].set_xlabel('Intertidal coverage [%]')
axs[1].set_ylabel('Number of tiles [-]')
axs[1].set_title('')

fig.tight_layout()

In [ ]:
# Figures
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
centers = [[113.8, -25], [-1.3, 46.1]]
sizes = [1, 1]

for idx in range(len(gdf_aois)):
    # Get square box
    center = centers[idx]
    size = sizes[idx]
    bounds = [center[0]-size/2, center[1]-size/2, center[0]+size/2, center[1]+size/2]
    gdf_box = gpd.GeoDataFrame({'geometry': [box(*bounds)]}, crs=crs)

    # Intersect with aoi
    gdf_tiles_aoi = gdf_tiles[gdf_tiles.intersects(gdf_box.unary_union)]
    gdf_mask_aoi = gdf_mask[gdf_mask.intersects(gdf_box.unary_union)]

    # Morphological erosion-dilation of mask with a buffer (1 pixel)
    gdf_mask_aoi_buffer = gdf_mask_aoi.copy()
    gdf_mask_aoi_buffer['geometry'] = gdf_mask_aoi_buffer['geometry'].buffer(-GEBCO_scale/2/111120) # divided by 2 to have half eroded on 2 sides (makes 1 in total) & fix for unequal buffer in EPSG3857: https://www.reddit.com/r/QGIS/comments/oo1jgh/buffer_points_on_epsg_4326_wgs_84/
    gdf_mask_aoi_buffer['geometry'] = gdf_mask_aoi_buffer['geometry'].buffer(GEBCO_scale/2/111120) # get it back
    gdf_mask_aoi_buffer = gdf_mask_aoi_buffer[~gdf_mask_aoi_buffer.is_empty] # removing the empty geometries

    # Plot aoi
    gdf_mask_aoi.plot(ax=axs[idx], color='blue', edgecolor='none', alpha=0.5)
    axs[idx].add_patch(mpatches.Rectangle((np.nan, np.nan), np.nan, np.nan, facecolor='blue', edgecolor='none', alpha=0.5, label='Original Processing area'))
    gdf_mask_aoi_buffer.plot(ax=axs[idx], color='red', edgecolor='none', alpha=0.5)
    axs[idx].add_patch(mpatches.Rectangle((np.nan, np.nan), np.nan, np.nan, facecolor='red', edgecolor='none', alpha=0.5, label='Refined processing area'))
    axs[idx].legend()

    # Set limits
    axs[idx].set_xlim(bounds[0], bounds[2])
    axs[idx].set_ylim(bounds[1], bounds[3])
    ctx.add_basemap(axs[idx], source=ctx.providers.OpenStreetMap.Mapnik, crs=crs)

    # Set labels
    axs[idx].set_xlabel('Longitude [deg]')
    axs[idx].set_ylabel('Latitude [deg]')

In [ ]:
# Packages
import rioxarray as rxr
import glob

# File paths
dir_path_tifs = r'p:\11209821-cmems-global-sdb\01_intertidal\02_data\05_calibrated\intertidal_improved_100m_upscaled_processed\03_clipped'
file_path_tifs = glob.glob(os.path.join(dir_path_tifs, '*.tif'))

# Read tifs
da_tif_ls = []
for file_path_tif in tqdm(file_path_tifs):
    # Read tif
    da_tif = rxr.open_rasterio(file_path_tif)

    # Reproject tif
    da_tif = da_tif.rio.reproject(dst_crs=crs)

    da_tif_ls.append(da_tif)

In [ ]:
da_tif_ls[0].rio.crs

In [ ]:
# Figures
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
centers = [[113.8, -25], [-1.3, 46.1]]
sizes = [1, 1]

for idx in range(len(gdf_aois)):
    # Get square box
    center = centers[idx]
    size = sizes[idx]
    bounds = [center[0]-size/2, center[1]-size/2, center[0]+size/2, center[1]+size/2]
    gdf_box = gpd.GeoDataFrame({'geometry': [box(*bounds)]}, crs=crs)

    # Intersect with aoi
    da_tif_aoi_ls = []
    for da_tif in da_tif_ls:
        if box(*da_tif.rio.bounds()).intersects(gdf_box.unary_union):
            da_tif_aoi = da_tif.rio.clip_box(*bounds)
            da_tif_aoi_ls.append(da_tif_aoi)
    gdf_tiles_aoi = gdf_tiles[gdf_tiles.intersects(gdf_box.unary_union)]
    gdf_stations_aoi = gdf_stations[gdf_stations.intersects(gdf_box.unary_union)]

    # Plot aoi
    for i, da_tif_aoi in enumerate(da_tif_aoi_ls):
        add_colorbar = True if i == 0 else False
        cbar_kwargs = {'label': 'Bathymetry [m]'} if i == 0 else {}
        da_tif_aoi.isel(band=0).plot.imshow(ax=axs[idx], add_colorbar=add_colorbar, vmin=-2, vmax=2, cmap='Spectral_r', cbar_kwargs=cbar_kwargs)
    axs[idx].set_title('')
    gdf_tiles_aoi.plot(ax=axs[idx], color='none', edgecolor='black', alpha=0.5)
    #gdf_stations_aoi.plot(ax=axs[idx], color='green', edgecolor='none', markersize=10, label='GTSM stations')

    # Set limits
    axs[idx].set_xlim(bounds[0], bounds[2])
    axs[idx].set_ylim(bounds[1], bounds[3])
    ctx.add_basemap(axs[idx], source=ctx.providers.OpenStreetMap.Mapnik, crs=crs, zorder=-1)

In [ ]:
gdf_aois

### Check time series data

In [ ]:
# Packages
import glob
import xarray as xr
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import contextily as ctx
import numpy as np
from tqdm import tqdm

In [ ]:
# File paths
file_path_his_ncs = r'p:\1230882-emodnet_hrsm\GTSMv3.0EMODnet\CMEMS_intertidal_SDB\*\output\gtsm_model_0000_his.nc'
file_path_his_ncs = glob.glob(file_path_his_ncs)
file_path_his_nc = file_path_his_ncs[2]
print(file_path_his_nc)

In [4]:
# Open his file
his = xr.open_dataset(file_path_his_nc)

In [ ]:
# Get subset of data
station_idxs_ls = [range(3715,3725), range(3725,3735), list(range(8957, 8962)) + [18456, 18457], list(range(3943, 3948)) + [19333, 19353, 19354, 21703]]

# Plot points
fig, axs = plt.subplots(1, len(station_idxs_ls), figsize=(16, 8))
for station_idxs, ax in zip(station_idxs_ls, axs):
    # Get subset of data
    his_stations = his.isel(stations=station_idxs)
    
    # Get points 
    points = [Point(x, y) for x, y in zip(his_stations.station_x_coordinate.values, his_stations.station_y_coordinate.values)]
    gdf = gpd.GeoDataFrame(geometry=points, crs='EPSG:4326')

    # Plot points
    gdf.plot(ax=ax, color='red', markersize=10)
    for i, point in gdf.iterrows():
        ax.text(point.geometry.x, point.geometry.y, station_idxs[i], fontsize=10, ha='center')
    ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, crs='EPSG:4326')
fig.tight_layout()

In [ ]:
# Plot time series
station_idxs = [range(3715,3725), range(3725,3735), list(range(8957, 8962)) + [18456, 18457], list(range(3943, 3948)) + [19333, 19353, 19354, 21703]]
station_idxs = [item for sublist in station_idxs for item in sublist]
# Time slice
time_slice = slice('2021-01-01', '2021-03-01')

# Plot time series
fig, axs = plt.subplots(int(np.ceil(len(station_idxs)/5)), 5, figsize=(16, 24))
axs = axs.flatten()
for i, station_idx in tqdm(enumerate(station_idxs), total=len(station_idxs)):
    # Get bedlevel and mean water level
    bedlevel = his.isel(stations=station_idx).bedlevel.values
    mean_waterlevel = his.isel(stations=station_idx).sel(time=time_slice).waterlevel.mean().values
    min_waterlevel = his.isel(stations=station_idx).sel(time=time_slice).waterlevel.min().values

    # Plot
    his.isel(stations=station_idx).sel(time=time_slice).waterlevel.plot(ax=axs[i])
    axs[i].axhline(y=bedlevel, color='r', linestyle='--', label='Bedlevel')
    axs[i].axhline(y=mean_waterlevel, color='g', linestyle='--', label='Mean water level')
    axs[i].set_title('Station {}\nBedlevel {:.2f} m\nMean water level {:.2f} m\nMin water level {:.2f} m'.format(station_idx, bedlevel, mean_waterlevel, min_waterlevel))
    axs[i].set_xlabel('Time')
    axs[i].set_ylabel('Water level [m]')
    axs[i].set_ylim(-2, 2)
    axs[i].legend()

    # Make background of box redish if absolute mean water level is higher than 0.05 m
    if np.abs(mean_waterlevel) > 0.05:
        axs[i].set_facecolor('mistyrose')
fig.tight_layout()

In [ ]:
# Plot time series
station_idxs = [3716, 18457]

# Time slice
time_slice = slice('2021-01-01', '2021-03-01')

# Plot time series
fig, axs = plt.subplots(1, len(station_idxs), figsize=(16, 6))
axs = axs.flatten()
for i, station_idx in tqdm(enumerate(station_idxs), total=len(station_idxs)):
    # Get bedlevel and mean water level
    bedlevel = his.isel(stations=station_idx).bedlevel.values
    mean_waterlevel = his.isel(stations=station_idx).sel(time=time_slice).waterlevel.mean().values
    min_waterlevel = his.isel(stations=station_idx).sel(time=time_slice).waterlevel.min().values

    # Plot
    his.isel(stations=station_idx).sel(time=time_slice).waterlevel.plot(ax=axs[i], label='Water level')
    axs[i].axhline(y=bedlevel, color='r', linestyle='--', label='Bed level')
    #axs[i].axhline(y=mean_waterlevel, color='g', linestyle='--', label='Mean water level')
    axs[i].set_title('Station {}'.format(station_idx))
    axs[i].set_xlabel('Time')
    axs[i].set_ylabel('Water level [m]')
    axs[i].set_ylim(-2, 2)
    axs[i].legend()
    axs[i].grid()
fig.tight_layout()

In [ ]:
his.isel(stations=200).waterlevel.mean().values

In [ ]:
his.isel(stations=0).waterlevel.mean().values

In [ ]:
df_tiles

In [ ]:
file_path = r'p:\1230882-emodnet_hrsm\GTSMv3.0EMODnet\CMEMS_intertidal_SDB\zarr_files\gtsm_model_his_2019.zarr'

ds = xr.open_zarr(file_path)